# Session 16: Judging a model: metrics, leakage and imbalance

Find the column that leaks the answer, work out precision and recall by hand, cross-validate the model with and without the leak, then draw the ROC curve and count the confusion matrix at one threshold.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. How rare are defaults?
Load `loans.csv` into `loans`. Store the share of loans that were repaid (`default == 0`) in `share_repaid`.

In [ ]:
import pandas as pd
import numpy as np

loans = pd.read_csv('loans.csv')
share_repaid = (loans.default == 0).mean()
print(round(share_repaid, 3))

In [ ]:
assert loans.shape == (8000, 18), 'Read loans.csv into loans. It should have 8000 rows and 18 columns'
assert 0.87 < share_repaid < 0.88, 'share_repaid should be the mean of a True or False test on default, near 0.875'
print('Task 1 correct')

## Task 2. Find the column that leaks
Correlate every number column with `default`. Keep the absolute values without `default` itself, sort largest first, and store the name of the top column in `leak`.

In [ ]:
corr = loans.corr(numeric_only=True)['default'].drop('default').abs().sort_values(ascending=False)
leak = corr.index[0]
print(corr.head(3).round(3))
print(leak)

In [ ]:
assert leak == 'collection_calls', 'Drop default itself, use absolute values, sort largest first, then take the first name of the index'
print('Task 2 correct')

## Task 3. Show why it leaks
Find the average `collection_calls` for repaid and for defaulted loans, in `calls_by_outcome`. Then store in `min_calls_defaulters` the fewest calls any defaulter had.

In [ ]:
calls_by_outcome = loans.groupby('default').collection_calls.mean()
min_calls_defaulters = loans[loans.default == 1].collection_calls.min()
print(calls_by_outcome.round(2))
print(min_calls_defaulters)

In [ ]:
assert calls_by_outcome[1] > 4 and calls_by_outcome[0] < 0.5, 'calls_by_outcome should show defaulters averaging over 4 calls and repaid loans under 0.5'
assert min_calls_defaulters == 1, 'min_calls_defaulters is the smallest collection_calls among defaulters only. Every defaulter has at least one call'
print('Task 3 correct')

## Task 4. Precision and recall by hand
A simple rule flags a loan when `previous_late_payments` is 1 or more. Using numpy, count `tp`, `fp`, `fn` and `tn` against `default`. Then compute `precision` and `recall`.

In [ ]:
flag = (loans.previous_late_payments >= 1).to_numpy()
actual = (loans.default == 1).to_numpy()
tp = int(np.sum(flag & actual))
fp = int(np.sum(flag & ~actual))
fn = int(np.sum(~flag & actual))
tn = int(np.sum(~flag & ~actual))
precision = tp / (tp + fp)
recall = tp / (tp + fn)
print(tp, fp, fn, tn)
print(round(precision, 3), round(recall, 3))

In [ ]:
assert tp + fp + fn + tn == 8000, 'The four counts must add up to 8000'
assert (tp, fn) == (421, 583), 'Check the flag and actual tests. tp is flagged and defaulted, fn is not flagged but defaulted'
assert abs(precision - 0.15) < 0.005, 'precision is tp divided by tp + fp'
assert abs(recall - 0.419) < 0.005, 'recall is tp divided by tp + fn'
print('Task 4 correct')

## Task 5. Cross-validate without the leak
Build `X_honest` from the ten safe number columns, and `y` from `default`. Cross-validate a pipeline with 5 folds and AUC scoring. Store the average in `auc_honest`.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

honest_cols = ['age', 'monthly_income_ugx', 'loan_amount_ugx', 'term_months',
               'previous_loans', 'previous_late_payments', 'group_loan',
               'savings_balance_ugx', 'has_mobile_money', 'distance_to_branch_km']
X_honest = loans[honest_cols]
y = loans.default
model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                      LogisticRegression(max_iter=1000, random_state=42))
scores = cross_val_score(model, X_honest, y, cv=5, scoring='roc_auc')
auc_honest = scores.mean()
print(scores.round(2), round(auc_honest, 3))

In [ ]:
assert 0.70 <= auc_honest <= 0.85, 'auc_honest should be between 0.70 and 0.85. Use scoring=\'roc_auc\' with cv=5 and take the mean'
print('Task 5 correct')

## Task 6. The same test with the leak
Add the leaking column to the features, cross-validate the same way, and store the average in `auc_leaky`.

In [ ]:
X_leaky = loans[honest_cols + [leak]]
scores_leaky = cross_val_score(model, X_leaky, y, cv=5, scoring='roc_auc')
auc_leaky = scores_leaky.mean()
print(round(auc_honest, 3), round(auc_leaky, 3))

In [ ]:
assert auc_leaky > 0.95, 'auc_leaky should be above 0.95. Add the leak column to the features, cross validate with AUC scoring, and take the mean'
assert auc_leaky > auc_honest + 0.1, 'The leaky AUC should be well above auc_honest. Did you add the leak column?'
print('Task 6 correct')

## Task 7. The ROC curve and the confusion matrix
Split `X_honest` and `y` with `test_size=0.25`, `stratify=y` and `random_state=42`. Fit the pipeline `model` on the training part, and get the chance of default for each test loan in `p`. Store the AUC in `auc` and draw the ROC curve. Then count the confusion matrix at a threshold of 0.2 into `tn_m`, `fp_m`, `fn_m` and `tp_m`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

X_train, X_test, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.25, stratify=y, random_state=42)
model.fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, p)
fpr, tpr, _ = roc_curve(y_test, p)
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False alarm rate')
plt.ylabel('Defaults caught')
plt.show()
cm = confusion_matrix(y_test, (p >= 0.2).astype(int))
tn_m, fp_m, fn_m, tp_m = cm.ravel()
print(round(auc, 3))
print(cm)

In [ ]:
assert 0.70 <= auc <= 0.85, 'auc should be near 0.76. Use roc_auc_score(y_test, p) with p from predict_proba, column 1'
assert tp_m + fn_m == int(y_test.sum()), 'tp_m and fn_m together are all the defaults in y_test'
assert tp_m + fp_m + fn_m + tn_m == 2000, 'The four counts cover 2000 test loans. Flag with p >= 0.2'
print('Task 7 correct')

## Task 8. Reflect
Write three sentences below. First, why would the leaky model fail on a loan approved today? Second, in a spreadsheet, which column would you check first for the same problem, and what question would you ask of it? Third, a missed default costs ten times a refused loan: do you watch precision or recall, and why?